# 1. Robust Uncertainty

## 1.0 Setup

*(code: import thư viện, đọc dữ liệu)*

### 1.0.1 Input data

**(1) `so_b2b` — dòng đơn bán B2B**
Mỗi dòng = 1 sản phẩm trong 1 đơn hàng gửi tới 1 điểm giao. Kỳ: tháng 1–11/2025. 24,084,062 dòng (26,790 tấn). Đã làm sạch: khử trùng lặp picking bị huỷ, bỏ sản phẩm lỗi, bỏ đơn Malaysia (Decision_Log 5.1–5.4).

| Trường | Kiểu | Ý nghĩa | Đơn vị | Dùng để |
|---|---|---|---|---|
| `OrderedTimestamp` | ngày giờ | Thời điểm khách đặt hàng | — | Xếp đơn vào tháng |
| `ProductID` | mã | Mã sản phẩm (SKU) | — | Tra kg/pallet của sản phẩm |
| `ShipToID` | mã | Điểm giao hàng (cửa hàng, nhà phân phối) | — | Tra điểm giao thuộc nhóm khách nào |
| `QtyOrderedInKG` | số | Lượng khách đặt (kể cả đơn sau đó bị huỷ) = **cầu** | kg | Tính cầu |

**(2) `product_master` — danh mục sản phẩm**
4,660 SKU (đã bỏ 514 SKU có kg/pallet bất thường, Decision_Log 5.4).

| Trường | Kiểu | Ý nghĩa | Đơn vị | Dùng để |
|---|---|---|---|---|
| `ProductID` | mã | Mã sản phẩm — khoá nối với `so_b2b` | — | Nối bảng |
| `KGPerPallet` | số | Bao nhiêu kg sản phẩm này thì đầy 1 pallet | kg/pallet | Đổi kg → pallet |

**(3) `shipto_group_map` — gán điểm giao vào nhóm khách**
34,430 điểm giao → 3,347 nhóm. Mỗi điểm giao thuộc đúng 1 nhóm; mỗi nhóm nằm trên 1 đảo (Decision_Log 4.6).

| Trường | Kiểu | Ý nghĩa | Đơn vị | Dùng để |
|---|---|---|---|---|
| `ShipToID` | mã | Điểm giao hàng — khoá nối với `so_b2b` | — | Nối bảng |
| `ShipToGroupID` | mã | Nhóm khách (các điểm giao gần nhau) = **nút cầu k** của mô hình | — | Gom cầu theo nhóm |

In [1]:
# ===== 1.0 SETUP =====
import pandas as pd                       # xử lý bảng
import numpy as np                        # tính toán
from pathlib import Path                  # ghép đường dẫn

pd.set_option("display.width", 200)       # in bảng rộng, không bị xuống dòng

IN = Path("../Model - Data used")         # data đã clean ở 2 notebook Validation (chỉ đọc)
OUT = Path("../Model Outputs")            # nơi lưu kết quả của các notebook model
OUT.mkdir(exist_ok=True)                  # tạo folder nếu chưa có

# Bảng gán khách -> nhóm: ShipToID (điểm giao hàng) -> ShipToGroupID (nhóm khách = nút cầu k của mô hình)
mp = pd.read_parquet(IN / "shipto_group_map.parquet")
G = sorted(mp["ShipToGroupID"].unique())                                 # tập nhóm khách, sắp xếp để thứ tự cố định
print(f"Ship-to: {len(mp):,} | nhóm khách: {len(G):,}")                  # kỳ vọng 34.430 ship-to, 3.347 nhóm

Ship-to: 34,430 | nhóm khách: 3,347


## 1.1 Theoretical background

### 1.1.1 Demand uncertainty in distribution network design

*(nội dung: vì sao cầu bất định; 3 cách xử lý: tất định / ngẫu nhiên (stochastic) / robust)*

### 1.1.2 Robust optimization approaches

*(nội dung: Soyster (1973) → Ben-Tal & Nemirovski → Bertsimas & Sim (2004); lý do chọn B&S: không cần phân phối xác suất, mô hình vẫn tuyến tính, có ngân sách Γ)*

### 1.1.3 Bertsimas & Sim uncertainty set and budget Γ

*(nội dung: khoảng [d̄, d̄ + d̂], ý nghĩa của Γ, "price of robustness")*

## 1.2 Demand data construction

### 1.2.1 From order lines to pallets

Mạng lưới kho tính chi phí vận chuyển và sức chứa theo **pallet**, không theo kg (Decision_Log 2.3). Mỗi sản phẩm có mật độ khác nhau (hàng nhẹ, cồng kềnh thì 1 pallet chứa ít kg), nên phải đổi kg → pallet **theo từng SKU trước**, rồi mới cộng lên:

$$q^{pallet}_{\ell} = \frac{q^{kg}_{\ell}}{\text{KGPerPallet}_{p(\ell)}}$$

- $\ell$: một dòng đơn trong `so_b2b`; $p(\ell)$: sản phẩm của dòng đó
- $q^{kg}_{\ell}$: lượng đặt (`QtyOrderedInKG`) — cầu theo Decision_Log 5.1
- $\text{KGPerPallet}_p$: số kg sản phẩm $p$ để đầy 1 pallet (`product_master`)

Ví dụ: 300 kg kem dưỡng (150 kg/pallet) = 2,0 pallet; 300 kg dầu gội (600 kg/pallet) = 0,5 pallet. Cộng kg trước rồi chia một hệ số chung sẽ sai.

In [2]:
# ===== 1.2.1 KG -> PALLET THEO TỪNG SẢN PHẨM =====
# Đọc đơn hàng: file 24 triệu dòng -> chỉ đọc 4 cột cần (xem bảng 1.0.1)
so = pd.read_parquet(IN / "so_b2b.parquet",
                     columns=["OrderedTimestamp", "ProductID", "ShipToID", "QtyOrderedInKG"])
pm = pd.read_parquet(IN / "product_master.parquet", columns=["ProductID", "KGPerPallet"])
print(f"Dòng đơn: {len(so):,} | tấn đặt: {so['QtyOrderedInKG'].sum()/1000:,.0f}")       # kỳ vọng 24.084.062 dòng, 26.790 tấn

# Bảng tra: mã sản phẩm -> kg/pallet
kgpp = pm.set_index("ProductID")["KGPerPallet"]

# Mỗi dòng đơn: pallet = kg đặt ÷ kg/pallet của chính sản phẩm đó
# (.astype(str) để mã sản phẩm cùng kiểu với bảng tra trước khi .map)
so["Pallet"] = so["QtyOrderedInKG"] / so["ProductID"].astype(str).map(kgpp)

# Kiểm tra: dòng nào không tìm được kg/pallet (sản phẩm không có trong master) -> báo lỗi, dừng
assert so["Pallet"].notna().all(), "Có sản phẩm trong đơn không có KGPerPallet"

# Tóm tắt: tổng pallet và kg bình quân mỗi pallet (hàng personal care nhẹ, cồng kềnh)
tong_pallet = so["Pallet"].sum()
print(f"Tổng pallet đặt (11 tháng): {tong_pallet:,.0f} | kg bình quân / pallet: "
      f"{so['QtyOrderedInKG'].sum() / tong_pallet:,.0f}")

Dòng đơn: 24,084,062 | tấn đặt: 26,790
Tổng pallet đặt (11 tháng): 109,406 | kg bình quân / pallet: 245


### 1.2.2 Aggregation to demand groups × month

*(nội dung + code: gom theo nhóm × tháng, tháng không đặt = 0)*

## 1.3 Uncertainty parameters

### 1.3.1 Nominal demand d̄ and maximum deviation d̂

*(nội dung + code: định nghĩa, vì sao chọn max − mean)*

### 1.3.2 Descriptive statistics

*(code: phân bố d̄, tỷ lệ d̂/d̄, số tháng có cầu)*

### 1.3.3 Export

*(code: lưu demand_robust, demand_monthly)*

## 1.4 Empirical evidence of demand uncertainty (industry characteristics)

### 1.4.1 Seasonality

*(code: chỉ số mùa vụ theo tháng)*

### 1.4.2 Group-level volatility

*(code: CV theo nhóm)*

### 1.4.3 Non-simultaneous peaks

*(code: tổng đỉnh từng nhóm so với đỉnh toàn quốc → lý do chọn tập bất định có ngân sách thay vì Soyster)*

### 1.4.4 Demand concentration and fragmentation

*(code: tập trung theo nhóm, số ship-to, kích thước dòng đơn)*

### 1.4.5 Product characteristics

*(code: SKU đuôi dài, kg/pallet)*

## 1.5 Implications for the model

*(nội dung: bảng đặc tính quan sát được → thành phần mô hình; đầu vào để chọn Γ ở notebook 2)*